# ED-Alert
### ELET – Early Leavers from Education and Training

In [1]:
%pip install eurostat
import eurostat
from pathlib import Path
import pandas as pd

Note: you may need to restart the kernel to use updated packages.


### ELET

In [7]:

df_elet = eurostat.get_data_df("edat_lfse_14")
print("---Dataset descargado---")

print("---Dataset ELET total---")
print(df_elet.head())

df_elet.columns
df_elet.to_csv("../data/03_elet.csv")

años = [str(y) for y in range(2000, 2026)]
df_comparativa = df_elet[
    (df_elet[r"geo\TIME_PERIOD"].isin(["ES", "EU27_2020"])) &
    (df_elet["sex"].isin(["T", "M", "F"])) &
    (df_elet["wstatus"] == "POP")
][["sex", r"geo\TIME_PERIOD"] + años].copy()

print("\n---Dataset ELET ESP + media europea---")
print(df_comparativa.head(6))

---Dataset descargado---
---Dataset ELET total---
  freq sex wstatus     age unit geo\TIME_PERIOD  1992  1993  1994  1995  ...  \
0    A   F     EMP  Y18-24   PC              AT   NaN   NaN   NaN   NaN  ...   
1    A   F     EMP  Y18-24   PC              BA   NaN   NaN   NaN   NaN  ...   
2    A   F     EMP  Y18-24   PC              BE   NaN   NaN   NaN   NaN  ...   
3    A   F     EMP  Y18-24   PC              BG   NaN   NaN   NaN   NaN  ...   
4    A   F     EMP  Y18-24   PC              CH   NaN   NaN   NaN   NaN  ...   

   2016  2017  2018  2019  2020  2021  2022  2023  2024  2025  
0   2.5   2.7   3.0   2.7   2.8   2.8   3.6   3.9   3.2   3.0  
1   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   0.8  
2   2.7   2.5   2.6   2.4   2.0   1.7   1.5   1.4   1.7   1.4  
3   2.0   2.5   2.2   2.9   2.4   2.2   2.5   1.7   NaN   NaN  
4   2.8   1.9   1.9   2.3   1.9   2.2   2.2   2.2   2.2   2.4  

[5 rows x 40 columns]

---Dataset ELET ESP + media europea---
    sex geo\TIME_PERIO

In [8]:
años = [str(y) for y in range(2000, 2026)]

df_comparativa = df_elet[
    (df_elet["geo\\TIME_PERIOD"].isin(["ES", "EU27_2020"])) &
    (df_elet["sex"].isin(["T", "M", "F"])) &  # T=total, M=hombres, F=mujeres
    (df_elet["wstatus"] == "POP")
][["sex", "geo\\TIME_PERIOD"] + años].copy()

print(df_comparativa)

    sex geo\TIME_PERIOD  2000  2001  2002  2003  2004  2005  2006  2007  ...  \
126   F              ES  23.2  23.1  24.3  24.8  25.0  24.7  23.6  24.7  ...   
127   F       EU27_2020   NaN   NaN  14.6  14.2  13.5  13.4  12.9  12.4  ...   
316   M              ES  35.0  36.0  37.2  38.4  39.0  37.0  36.7  36.6  ...   
317   M       EU27_2020   NaN   NaN  19.1  18.6  18.4  17.7  17.4  16.9  ...   
506   T              ES  29.1  29.7  30.9  31.7  32.2  31.0  30.3  30.8  ...   
507   T       EU27_2020   NaN   NaN  16.9  16.4  16.0  15.6  15.2  14.7  ...   

     2016  2017  2018  2019  2020  2021  2022  2023  2024  2025  
126  15.1  14.5  14.0  13.0  11.6   9.7  11.1  11.3  10.0   9.5  
127   9.1   8.9   8.7   8.4   8.1   7.9   8.0   7.7   7.7   7.5  
316  22.7  21.8  21.7  21.4  20.2  16.7  16.7  16.0  15.8  15.9  
317  12.1  12.1  12.1  11.8  11.9  11.5  11.1  11.3  11.0  10.6  
506  19.0  18.3  17.9  17.3  16.0  13.3  13.9  13.7  13.0  12.8  
507  10.6  10.5  10.5  10.1  10.0   9.7   9

In [9]:
# Transformar a formato largo para Tableau
df_elet_long = df_comparativa.melt(
    id_vars=["sex", "geo\\TIME_PERIOD"],
    value_vars=[str(y) for y in range(2000, 2026)],
    var_name="año",
    value_name="tasa_abandono"
)

# Renombrar columnas
df_elet_long = df_elet_long.rename(columns={
    "geo\\TIME_PERIOD": "pais",
    "sex": "sexo"
})

# Mapear códigos a etiquetas legibles
df_elet_long["pais"] = df_elet_long["pais"].map({
    "ES": "España",
    "EU27_2020": "Media UE27"
})

df_elet_long["sexo"] = df_elet_long["sexo"].map({
    "T": "Total",
    "M": "Hombres",
    "F": "Mujeres"
})

# Convertir año a entero
df_elet_long["año"] = df_elet_long["año"].astype(int)

# Eliminar nulos
df_elet_long = df_elet_long.dropna(subset=["tasa_abandono"])

print(df_elet_long.head(10))
print(df_elet_long.shape)

# Exportar
df_elet_long.to_csv("../data/03_elet_long.csv", index=False)

       sexo        pais   año  tasa_abandono
0   Mujeres      España  2000           23.2
2   Hombres      España  2000           35.0
4     Total      España  2000           29.1
6   Mujeres      España  2001           23.1
8   Hombres      España  2001           36.0
10    Total      España  2001           29.7
12  Mujeres      España  2002           24.3
13  Mujeres  Media UE27  2002           14.6
14  Hombres      España  2002           37.2
15  Hombres  Media UE27  2002           19.1
(150, 4)
